# GitLab Merge Request Enricher

This notebook enriches merge request records in `gitlab_mr_enricher.xlsx` using GitLab API metadata,
then assigns each MR to the pod / crew the author belonged to **on the day the MR was merged**.

Workflow:
1. Read token securely from user input
2. Read `gitlab_mr_enricher.xlsx` from notebook folder
3. Resolve project name and path from `target_project_id`
4. Resolve author details from `target_project_id` plus IID (with ID fallback)
5. Extract `author_gpn` and `merged_month_year`
6. Build historical pod/crew membership intervals per GPN from `crew_pod_members_historical.xlsx`
7. Point-in-time match each MR to the pod/crew active on its merge date
8. Write enriched output XLSX with all original and new columns, plus per-pod / per-crew chart sheets

---
### Fixes applied in this version
**FIX 1 - Author charts:** every author is now plotted (long lists are split into several
stacked charts of `MAX_AUTHORS_PER_CHART` each, and the category axis is forced to show
one label per author), and the bars are sorted **descending** on average lead time in hours
(largest at the top).

**FIX 2 - pod_name / crew_name were coming out as N/A for every row.** Root cause: GPNs in the
pod file are read by pandas as floats, so they stringified as `'43246145.0'`, while GPNs parsed
out of the GitLab author name are plain digits like `'43673730'` - so the join matched 0 rows.
Both sides are now pushed through `normalize_gpn()`. On top of that, the membership intervals
are now derived properly per GPN (each start date runs until the day before that GPN's next
start date, last interval runs open-ended), so an MR merged on any date lands in the correct
pod/crew for that date.

## Section 1: Set Constants, Imports, and Notebook Options

In [ ]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from pathlib import Path
import time
from typing import Dict, Tuple, Optional
from getpass import getpass

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

GITLAB_URL = 'https://devcloud.ubs.net'
INPUT_XLSX_NAME = 'gitlab_mr_enricher.xlsx'
POD_FILE_NAME = 'crew_pod_members_historical.xlsx'
OUTPUT_SUFFIX = '_output_enriched'

MAX_WORKERS = 20
REQUEST_TIMEOUT = 20

# ---------------------------------------------------------------------------
# Pod / crew matching options
# ---------------------------------------------------------------------------
# The pod file only records START dates. The end of each membership is derived as
# (next start date for that GPN) - 1 day. The final membership stays open-ended.
#
# BACKFILL_EARLIEST_MEMBERSHIP: if an MR was merged BEFORE the earliest start date
# recorded for that GPN, treat the earliest membership as if it also covered that
# earlier period. Set to False if you want strict matching (those MRs become N/A).
BACKFILL_EARLIEST_MEMBERSHIP = True

# Open-ended / backfilled interval bounds
FAR_FUTURE = pd.Timestamp('2099-12-31')
FAR_PAST = pd.Timestamp('1990-01-01')

# ---------------------------------------------------------------------------
# Chart options
# ---------------------------------------------------------------------------
# Excel silently drops category labels when a bar chart has too many categories, and a
# very tall chart gets clipped. So author charts are split into chunks of this size, with
# every chunk showing every label and every data value.
MAX_AUTHORS_PER_CHART = 50
CHART_ROW_HEIGHT_PX = 20     # vertical space per author bar
CHART_WIDTH_PX = 900

start_time = time.time()


def elapsed_seconds() -> float:
    return time.time() - start_time

## Section 2: Read User Input (Token) and Fixed XLSX Path

In [ ]:
gitlab_token = getpass('Enter GitLab Personal Access Token: ').strip()
if not gitlab_token:
    raise ValueError('GitLab token is required.')

notebook_dir = Path.cwd().resolve()
input_path = notebook_dir / INPUT_XLSX_NAME
if not input_path.exists():
    raise FileNotFoundError(
        f'Input file not found: {input_path}. Place {INPUT_XLSX_NAME} in the same folder as this notebook.'
    )

print(f'Using GitLab URL: {GITLAB_URL}')
print(f'Using input file: {input_path}')

## Section 3: Load XLSX and Validate Required Columns

In [ ]:
print('Loading XLSX...')
df = pd.read_excel(input_path)
print(f'Loaded rows: {len(df)}')


def _norm_col_name(col) -> str:
    """Lowercase and strip everything that is not a letter or digit."""
    return re.sub(r'[^a-z0-9]+', '', str(col).strip().lower())


alias_map = {
    'ID': ['id', 'mrid', 'mergerequestid', 'mergeid', 'requestid'],
    'IID': ['iid', 'mriid', 'mergerequestiid', 'mergeiid', 'requestiid'],
    'target_project_id': ['targetprojectid', 'projectid', 'targetproject', 'targetprojid'],
    'merged_at': ['mergedat', 'mergeat', 'mergeddate', 'mergeddatetime', 'mergetime'],
}

normalized_to_actual = {_norm_col_name(c): c for c in df.columns}
rename_map = {}
for canonical, aliases in alias_map.items():
    if canonical in df.columns:
        continue
    for alias in aliases:
        actual = normalized_to_actual.get(alias)
        if actual:
            rename_map[actual] = canonical
            break

if rename_map:
    df = df.rename(columns=rename_map)
    print(f'Auto-renamed columns: {rename_map}')

required_cols = ['ID', 'IID', 'target_project_id', 'merged_at']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    print(f'Available columns: {list(df.columns)}')
    raise ValueError(f'Missing required columns after auto-mapping: {missing_cols}')

# Normalize datatypes and keep original row order
for col in ['ID', 'IID', 'target_project_id']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

invalid_proj_id_rows = int(df[['IID', 'target_project_id']].isna().any(axis=1).sum())
if invalid_proj_id_rows:
    print(f'Rows with invalid IID/target_project_id: {invalid_proj_id_rows}')

df['ID'] = df['ID'].astype('Int64')
df['IID'] = df['IID'].astype('Int64')
df['target_project_id'] = df['target_project_id'].astype('Int64')
df['merged_at'] = pd.to_datetime(df['merged_at'], errors='coerce', utc=True)

# Build unique lookup keys to reduce API calls (ID kept as fallback key)
df_keys = df[['target_project_id', 'IID', 'ID']].dropna(subset=['target_project_id', 'IID']).drop_duplicates().copy()
print(f'Unique MR lookup keys: {len(df_keys)}')

## Section 4: Create GitLab API Client with Reusable Session

In [ ]:
class GitLabClient:
    def __init__(self, base_url: str, token: str, timeout: int = REQUEST_TIMEOUT):
        self.base_url = base_url.rstrip('/')
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update({'PRIVATE-TOKEN': token})

        retry = Retry(
            total=4,
            backoff_factor=0.4,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=['GET'],
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=100, pool_maxsize=100)
        self.session.mount('https://', adapter)
        self.session.mount('http://', adapter)

        self.cache: Dict[str, dict] = {}
        self.api_success = 0
        self.api_failed = 0

    def _get_json(self, endpoint: str, use_cache: bool = True) -> Optional[dict]:
        cache_key = endpoint
        if use_cache and cache_key in self.cache:
            return self.cache[cache_key]

        url = f'{self.base_url}/api/v4{endpoint}'
        try:
            resp = self.session.get(url, timeout=self.timeout)
            resp.raise_for_status()
            payload = resp.json()
            if use_cache:
                self.cache[cache_key] = payload
            self.api_success += 1
            return payload
        except requests.RequestException:
            self.api_failed += 1
            return None

    def get_project(self, project_id: int) -> Optional[dict]:
        return self._get_json(f'/projects/{project_id}')

    def get_mr_by_iid(self, project_id: int, mr_iid: int) -> Optional[dict]:
        return self._get_json(f'/projects/{project_id}/merge_requests/{mr_iid}')

    def get_mr_by_id(self, mr_id: int) -> Optional[dict]:
        return self._get_json(f'/merge_requests/{mr_id}')

    def stats(self) -> Dict[str, int]:
        return {
            'success': self.api_success,
            'failed': self.api_failed,
            'cached': len(self.cache),
        }


gl_client = GitLabClient(GITLAB_URL, gitlab_token)
print('GitLab client initialized.')

## Section 5: Resolve Project Metadata from target_project_id (Cached + Parallel)

In [ ]:
print('Resolving project metadata...')

unique_project_ids = sorted(df['target_project_id'].dropna().astype(int).unique().tolist())
project_meta: Dict[int, Dict[str, str]] = {}


def fetch_project(pid: int) -> Tuple[int, Dict[str, str]]:
    payload = gl_client.get_project(pid)
    if not payload:
        return pid, {'project_name': 'N/A', 'project_path': 'N/A', 'group_path': 'N/A'}
    path_with_ns = payload.get('path_with_namespace', payload.get('path', 'N/A'))
    # group_path is the namespace portion (everything before the last "/" segment)
    group = '/'.join(path_with_ns.split('/')[:-1]) if '/' in path_with_ns else 'N/A'
    return pid, {
        'project_name': payload.get('name', 'N/A'),
        'project_path': f'{GITLAB_URL}/{path_with_ns}',
        'group_path': f'{GITLAB_URL}/{group}' if group != 'N/A' else 'N/A',
    }


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(fetch_project, pid) for pid in unique_project_ids]
    for future in as_completed(futures):
        pid, value = future.result()
        project_meta[pid] = value

project_meta_df = pd.DataFrame.from_dict(project_meta, orient='index').reset_index()
project_meta_df.columns = ['target_project_id', 'project_name', 'project_path', 'group_path']
project_meta_df['target_project_id'] = project_meta_df['target_project_id'].astype('Int64')

print(f'Project metadata resolved for {len(project_meta_df)} unique projects.')

## Section 6: Resolve MR Author Metadata from ID and IID (Parallel + Retry)

In [ ]:
print('Resolving merge request author metadata...')

mr_meta: Dict[Tuple[int, int], Dict[str, str]] = {}


def fetch_mr_author(project_id: int, mr_iid: int, mr_id_fallback: int) -> Tuple[Tuple[int, int], Dict[str, str]]:
    payload = gl_client.get_mr_by_iid(project_id, mr_iid)

    # Fallback: if IID endpoint is missing or id mismatch, try global MR id.
    if (not payload or ('id' in payload and mr_id_fallback > 0
                        and int(payload.get('id', -1)) != int(mr_id_fallback))) and mr_id_fallback > 0:
        fallback = gl_client.get_mr_by_id(mr_id_fallback)
        if fallback:
            payload = fallback

    if not payload:
        return (project_id, mr_iid), {'author_name': 'N/A'}

    author = payload.get('author') or {}
    return (project_id, mr_iid), {'author_name': author.get('name', 'N/A')}


lookup_rows = []
for row in df_keys.itertuples(index=False):
    project_id = int(row.target_project_id)
    mr_iid = int(row.IID)
    mr_id_fallback = int(row.ID) if pd.notna(row.ID) else -1
    lookup_rows.append((project_id, mr_iid, mr_id_fallback))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(fetch_mr_author, pid, iid, mid) for pid, iid, mid in lookup_rows]
    for future in as_completed(futures):
        key, value = future.result()
        mr_meta[key] = value

mr_meta_df = pd.DataFrame(
    [
        {
            'target_project_id': k[0],
            'IID': k[1],
            'author_name': v.get('author_name', 'N/A'),
        }
        for k, v in mr_meta.items()
    ]
)

mr_meta_df['target_project_id'] = mr_meta_df['target_project_id'].astype('Int64')
mr_meta_df['IID'] = mr_meta_df['IID'].astype('Int64')

print(f'Author metadata resolved for {len(mr_meta_df)} unique MRs.')

## Section 7: Extract author_gpn from Author Name (+ GPN Normalizer)

In [ ]:
print('Extracting author_gpn from author_name...')


def split_author_and_gpn(author_name: str) -> Tuple[str, str]:
    if not isinstance(author_name, str) or not author_name.strip() or author_name == 'N/A':
        return 'N/A', 'N/A'

    text = author_name.strip()

    # Case 1: Name (gpn) e.g. "John Doe (43673730)"
    m_paren = re.search(r'^(.*?)\s*\(([^()]+)\)\s*$', text)
    if m_paren:
        return m_paren.group(1).strip(), m_paren.group(2).strip()

    # Case 2: Name followed by numeric GPN e.g. "Chavan Yogesh 43673730"
    m_digits = re.search(r'^(.*?)\s+(\d{5,10})\s*$', text)
    if m_digits:
        return m_digits.group(1).strip(), m_digits.group(2).strip()

    # Case 3: Name followed by t/u prefixed GPN e.g. "John Doe t741348"
    m_end = re.search(r'^(.*?)[\s_-]+([tu]\d{5,9})\s*$', text, flags=re.IGNORECASE)
    if m_end:
        return m_end.group(1).strip(), m_end.group(2).strip()

    return text, 'N/A'


def normalize_gpn(value) -> str:
    """
    FIX 2 (root cause): return a GPN as a plain digit string so both sides of the join match.

    pandas reads the GPN column of crew_pod_members_historical.xlsx as float64, so
    str(gpn) produced '43246145.0' while GPNs parsed from GitLab author names are
    '43673730'. Nothing ever matched. This collapses both forms to '43246145'.
    """
    if value is None:
        return ''
    if isinstance(value, float):
        if pd.isna(value):
            return ''
        if float(value).is_integer():
            return str(int(value))
    if isinstance(value, int):
        return str(value)

    s = str(value).strip()
    if s == '' or s.lower() in ('nan', 'none', 'n/a', 'na', '<na>'):
        return ''

    # '43246145.0' -> '43246145'
    m = re.match(r'^(\d+)\.0+$', s)
    if m:
        return m.group(1)

    # Strip any t/u prefix, spaces, or stray punctuation -> digits only
    digits = re.sub(r'\D', '', s)
    return digits


# Quick self-check of the normalizer
for _sample in ['43246145.0', 43246145.0, '43673730', ' 43725929 ', 't741348', 'N/A', None]:
    print(f'  normalize_gpn({_sample!r}) -> {normalize_gpn(_sample)!r}')

## Section 8: Create merged_month_year from merged_at

In [ ]:
print('Building merged_month_year from merged_at...')
df['merged_month_year'] = df['merged_at'].dt.strftime('%Y-%m').fillna('N/A')
print('merged_month_year ready.')

## Section 9: Assemble Enriched Frame and Load Pod Membership

In [ ]:
print('Assembling final dataframe...')

# Merge project metadata
df_enriched = df.merge(project_meta_df, on='target_project_id', how='left')

# Merge MR author metadata using project + IID
df_enriched = df_enriched.merge(mr_meta_df, on=['target_project_id', 'IID'], how='left', suffixes=('', '_mr'))
if 'author_name_mr' in df_enriched.columns:
    df_enriched['author_name'] = df_enriched['author_name_mr']
    df_enriched = df_enriched.drop(columns=['author_name_mr'])

# Normalize missing values
for _c in ['project_name', 'project_path', 'group_path', 'author_name']:
    df_enriched[_c] = df_enriched[_c].fillna('N/A')

# Split author and gpn
author_split = df_enriched['author_name'].apply(split_author_and_gpn)
df_enriched['author_name'] = author_split.apply(lambda x: x[0])
df_enriched['author_gpn'] = author_split.apply(lambda x: x[1])

# Identify the lead time column from input data (robust to spaces vs underscores)
LEAD_TIME_COL = None
for col in df_enriched.columns:
    if 'totalleadtimeinhours' in re.sub(r'[^a-z0-9]+', '', str(col).lower()):
        LEAD_TIME_COL = col
        break

if LEAD_TIME_COL:
    df_enriched[LEAD_TIME_COL] = pd.to_numeric(df_enriched[LEAD_TIME_COL], errors='coerce')
    print(f'Using lead time column: "{LEAD_TIME_COL}"')
else:
    print('WARNING: No "total_lead_time_in_hours" column found in input. Lead time graphs will be empty.')

# merged_at date only, timezone naive -- this is the key used for point-in-time pod matching
_merged = pd.to_datetime(df_enriched['merged_at'], errors='coerce', utc=True)
df_enriched['merged_date'] = _merged.dt.tz_convert(None).dt.normalize()

new_cols = ['project_name', 'project_path', 'group_path', 'author_name', 'author_gpn', 'merged_month_year']
original_cols = [c for c in df.columns if c not in new_cols and c != 'merged_month_year']
output_df = df_enriched[original_cols + new_cols].copy()
output_df['merged_date'] = df_enriched['merged_date']

# Drop any remaining duplicate columns
output_df = output_df.loc[:, ~output_df.columns.duplicated()]

# Remove lead_time_days and lead_time_hours if they exist
drop_cols = [c for c in output_df.columns if c in ('lead_time_days', 'lead_time_hours')]
if drop_cols:
    output_df = output_df.drop(columns=drop_cols)

# Excel cannot store timezone-aware datetimes; convert them to timezone-naive.
tz_cols = output_df.select_dtypes(include=['datetimetz']).columns.tolist()
for col in tz_cols:
    output_df[col] = output_df[col].dt.tz_convert(None)
if tz_cols:
    print(f'Converted timezone-aware columns to naive for Excel: {tz_cols}')

print(f'Enriched dataframe ready: {len(output_df)} rows, {len(output_df.columns)} columns')

# --- Load crew_pod_members_historical.xlsx ------------------------------------
pod_path = notebook_dir / POD_FILE_NAME
if not pod_path.exists():
    print(f'WARNING: {POD_FILE_NAME} not found. Pod/crew matching will be skipped.')
    pod_df = pd.DataFrame()
else:
    print(f'Loading {POD_FILE_NAME}...')
    pod_df = pd.read_excel(pod_path)
    print(f'Loaded {len(pod_df)} pod membership rows')

    # Normalize column names for robustness
    pod_df.columns = [str(c).strip() for c in pod_df.columns]

    def _pick(pred, cols):
        hits = [c for c in cols if pred(c)]
        return hits[0] if hits else None

    _cols = list(pod_df.columns)
    L1_TYPE = _pick(lambda c: 'l1' in c.lower() and 'type' in c.lower(), _cols)
    # Prefer "Allocation (%)" over "Total Allocation (%)"
    _alloc_all = [c for c in _cols if 'allocation' in c.lower() and '%' in c]
    _alloc_pref = [c for c in _alloc_all if 'total' not in c.lower()]
    ALLOC = (_alloc_pref or _alloc_all or [None])[0]
    GPN_COL = _pick(lambda c: c.strip().upper() == 'GPN', _cols) or _pick(lambda c: 'gpn' in c.lower(), _cols)
    START_DATE = _pick(lambda c: 'start' in c.lower() and 'date' in c.lower(), _cols)
    L1_NAME = _pick(lambda c: 'l1' in c.lower() and 'name' in c.lower(), _cols)
    L0_NAME = _pick(lambda c: 'l0' in c.lower() and 'name' in c.lower(), _cols)

    print(f'  Detected columns: L1type={L1_TYPE}, Alloc={ALLOC}, GPN={GPN_COL}, '
          f'Start={START_DATE}, L1Name={L1_NAME}, L0Name={L0_NAME}')

    required_pod_cols = [L1_TYPE, ALLOC, GPN_COL, START_DATE, L1_NAME, L0_NAME]
    if any(c is None for c in required_pod_cols):
        print('ERROR: Missing required columns in pod file. Pod matching skipped.')
        pod_df = pd.DataFrame()
    else:
        # Filter: L1 Type == 'Pod' AND Allocation (%) > 0
        pod_df[ALLOC] = pd.to_numeric(pod_df[ALLOC], errors='coerce').fillna(0)
        pod_df = pod_df[(pod_df[L1_TYPE].astype(str).str.strip().str.lower() == 'pod') & (pod_df[ALLOC] > 0)].copy()

        pod_df['gpn_norm'] = pod_df[GPN_COL].apply(normalize_gpn)          # FIX 2
        pod_df[START_DATE] = pd.to_datetime(pod_df[START_DATE], errors='coerce').dt.normalize()
        pod_df['pod_name'] = pod_df[L1_NAME].astype(str).str.strip()       # L1 Name -> pod
        pod_df['crew_name'] = pod_df[L0_NAME].astype(str).str.strip()      # L0 Name -> crew

        print(f'  Filtered pod memberships (L1 Type=Pod, Allocation>0): {len(pod_df)} rows')

## Section 10: Match MRs to Pods and Crews (point-in-time)

For every GPN, the rows in the pod file are sorted by start date. Each start date opens a
membership that runs until **the day before that GPN's next start date**; the newest start date
stays open-ended. An MR is then assigned the pod/crew whose interval contains its `merged_date`.

In [ ]:
print('Matching MRs to pods/crews using crew_pod_members_historical.xlsx...')

if pod_df.empty:
    output_df['pod_name'] = 'N/A'
    output_df['crew_name'] = 'N/A'
    pod_output_df = output_df.copy()
else:
    # ---- Build the membership lookup ---------------------------------------
    pod_lookup = pod_df[['gpn_norm', 'pod_name', 'crew_name']].copy()
    pod_lookup['start_date'] = pod_df[START_DATE]
    pod_lookup = pod_lookup.rename(columns={'gpn_norm': 'gpn'})

    # Drop rows with invalid GPN or start date
    pod_lookup = pod_lookup.dropna(subset=['start_date'])
    pod_lookup = pod_lookup[pod_lookup['gpn'] != ''].copy()
    pod_lookup = pod_lookup.drop_duplicates(subset=['gpn', 'start_date', 'pod_name', 'crew_name'])
    pod_lookup = pod_lookup.sort_values(['gpn', 'start_date']).reset_index(drop=True)

    # Derive end_date per GPN from that GPN's NEXT DISTINCT start date - 1 day.
    # (Using distinct start dates so a GPN sitting in two pods on the same start
    #  date does not truncate its own interval to zero length.)
    intervals = pod_lookup[['gpn', 'start_date']].drop_duplicates().sort_values(['gpn', 'start_date'])
    intervals['end_date'] = intervals.groupby('gpn')['start_date'].shift(-1) - pd.Timedelta(days=1)
    intervals['end_date'] = intervals['end_date'].fillna(FAR_FUTURE)   # newest membership is open ended
    intervals['effective_start'] = intervals['start_date']
    if BACKFILL_EARLIEST_MEMBERSHIP:
        first_rows = intervals.groupby('gpn', as_index=False).head(1).index
        intervals.loc[first_rows, 'effective_start'] = FAR_PAST

    pod_lookup = pod_lookup.merge(intervals, on=['gpn', 'start_date'], how='left')

    print(f'  Pod lookup table: {len(pod_lookup)} membership records '
          f'covering {pod_lookup["gpn"].nunique()} GPNs')

    # ---- Normalize author_gpn on the MR side (FIX 2) -----------------------
    output_df['_gpn_match'] = output_df['author_gpn'].apply(normalize_gpn)

    _pod_gpns = set(pod_lookup['gpn'])
    _mr_gpns = set(g for g in output_df['_gpn_match'] if g)
    print(f'  Sample GPNs in pod file: {sorted(list(_pod_gpns))[:5]}')
    print(f'  Sample GPNs in MR data : {sorted(list(_mr_gpns))[:5]}')
    print(f'  GPNs present in both   : {len(_pod_gpns & _mr_gpns)} '
          f'(of {len(_mr_gpns)} distinct MR authors)')

    # ---- Point-in-time match ------------------------------------------------
    mr_keys = output_df[['_gpn_match', 'merged_date']].drop_duplicates().copy()
    mr_keys.columns = ['gpn', 'merged_date']
    mr_keys = mr_keys[(mr_keys['gpn'] != '') & mr_keys['merged_date'].notna()].copy()

    matched = mr_keys.merge(pod_lookup, on='gpn', how='inner')
    matched = matched[
        (matched['merged_date'] >= matched['effective_start'])
        & (matched['merged_date'] <= matched['end_date'])
    ][['gpn', 'merged_date', 'pod_name', 'crew_name']].drop_duplicates()

    print(f'  Matched (gpn, merge date) -> pod records: {len(matched)}')

    # Show a sample of derived date ranges for debugging
    if len(pod_lookup):
        sample_gpn = sorted(_pod_gpns & _mr_gpns)[0] if (_pod_gpns & _mr_gpns) else pod_lookup['gpn'].iloc[0]
        sample_rows = pod_lookup[pod_lookup['gpn'] == sample_gpn][
            ['gpn', 'start_date', 'end_date', 'pod_name', 'crew_name']
        ]
        print(f'  Sample derived date ranges for GPN {sample_gpn}:')
        print(sample_rows.to_string(index=False))

    # Left-join all MR keys so unmatched stay visible
    all_assignments = mr_keys.merge(matched, on=['gpn', 'merged_date'], how='left')
    all_assignments['pod_name'] = all_assignments['pod_name'].fillna('N/A')
    all_assignments['crew_name'] = all_assignments['crew_name'].fillna('N/A')

    # Merge back to output_df. A GPN in two pods on the merge date produces one row per pod.
    pod_output_df = output_df.merge(
        all_assignments,
        left_on=['_gpn_match', 'merged_date'],
        right_on=['gpn', 'merged_date'],
        how='left',
    ).drop(columns=['gpn'], errors='ignore')

    pod_output_df['pod_name'] = pod_output_df['pod_name'].fillna('N/A')
    pod_output_df['crew_name'] = pod_output_df['crew_name'].fillna('N/A')

    pod_output_df = pod_output_df.drop(columns=['_gpn_match'], errors='ignore')
    output_df = output_df.drop(columns=['_gpn_match'], errors='ignore')

    # ---- Summary ------------------------------------------------------------
    pod_counts = pod_output_df['pod_name'].value_counts()
    print(f'Total rows after pod expansion: {len(pod_output_df)}')
    print(f'Unique pods: {pod_output_df["pod_name"].nunique()}')
    print(f'Unique crews: {pod_output_df["crew_name"].nunique()}')
    print(f'Rows with pod assigned: {(pod_output_df["pod_name"] != "N/A").sum()}')
    print(f'Rows without pod (N/A): {(pod_output_df["pod_name"] == "N/A").sum()}')
    print('MR distribution by pod:')
    print(pod_counts.to_string())

## Section 11: Write Output XLSX with Pod & Crew Worksheets and Charts

In [ ]:
print('Writing output XLSX with pod and crew worksheets...')

output_path = input_path.with_name(f'{input_path.stem}{OUTPUT_SUFFIX}.xlsx')

# Remove merged_date helper column from final output
write_cols = [c for c in pod_output_df.columns if c != 'merged_date']
write_cols = [c for c in write_cols if c not in ('lead_time_days', 'lead_time_hours')]
final_df = pod_output_df[write_cols].copy()

# Determine lead time column for graphs
LT_COL = LEAD_TIME_COL if LEAD_TIME_COL and LEAD_TIME_COL in pod_output_df.columns else None
if LT_COL:
    print(f'Using "{LT_COL}" for lead time graphs')
else:
    print('No lead time column available for graphs')


def add_pod_charts(workbook, writer, sheet_name, data_df):
    """
    Adds charts to a worksheet with data labels:
      1. Author vs avg lead time (hours) - bar, DESCENDING (largest at top), ALL authors
      2. Author vs MR count            - bar, DESCENDING (largest at top), ALL authors
      3. YYYY-MM vs avg lead time      - line with trendline
      4. YYYY-MM vs MR count           - line with trendline
    All charts are stacked vertically in the same worksheet.

    FIX 1: authors are sorted strictly descending on the plotted value, the category axis is
    forced to render one label per author (interval_unit / interval_tick = 1), and long author
    lists are split into several charts of MAX_AUTHORS_PER_CHART so nothing is clipped or skipped.
    """
    has_lt = LT_COL is not None and LT_COL in data_df.columns and data_df[LT_COL].notna().any()

    # Stats by author for lead time - ALL authors, sorted DESCENDING
    if has_lt:
        author_lt = data_df.groupby('author_name', as_index=False).agg(
            avg_lead_time_hours=(LT_COL, 'mean'),
        )
        author_lt['avg_lead_time_hours'] = author_lt['avg_lead_time_hours'].round(2)
        author_lt = author_lt.sort_values('avg_lead_time_hours', ascending=False).reset_index(drop=True)
    else:
        author_lt = pd.DataFrame(columns=['author_name', 'avg_lead_time_hours'])

    # Stats by author for MR count - ALL authors, sorted DESCENDING
    author_mr = data_df.groupby('author_name', as_index=False).agg(mr_count=('ID', 'count'))
    author_mr = author_mr.sort_values('mr_count', ascending=False).reset_index(drop=True)

    # Stats by month
    month_data = data_df[data_df['merged_month_year'] != 'N/A']
    if has_lt:
        month_stats = month_data.groupby('merged_month_year', as_index=False).agg(
            mr_count=('ID', 'count'),
            avg_lead_time_hours=(LT_COL, 'mean'),
        ).sort_values('merged_month_year')
        month_stats['avg_lead_time_hours'] = month_stats['avg_lead_time_hours'].round(2)
    else:
        month_stats = month_data.groupby('merged_month_year', as_index=False).agg(
            mr_count=('ID', 'count'),
        ).sort_values('merged_month_year')
        month_stats['avg_lead_time_hours'] = None

    # Write data tables to the sheet
    # Table 1: author lead time at col 0 (A:B)
    author_lt.to_excel(writer, sheet_name=sheet_name, index=False, startrow=0, startcol=0)
    # Table 2: author MR count at col 3 (D:E)
    author_mr.to_excel(writer, sheet_name=sheet_name, index=False, startrow=0, startcol=3)
    # Table 3: month stats at col 7 (H:J)
    month_offset_col = 7
    month_stats.to_excel(writer, sheet_name=sheet_name, index=False, startrow=0, startcol=month_offset_col)

    ws = writer.sheets[sheet_name]
    ws.set_column(0, 0, 32)
    ws.set_column(3, 3, 32)
    ws.set_column(month_offset_col, month_offset_col, 14)

    n_auth_lt = len(author_lt)
    n_auth_mr = len(author_mr)
    n_month = len(month_stats)

    # Charts start below the widest data table
    current_chart_row = max(n_auth_lt, n_auth_mr, n_month) + 3

    def add_author_bar_charts(n_items, cat_col, val_col, series_name, chart_title,
                              value_axis_name, num_fmt, row_cursor):
        """One chart per MAX_AUTHORS_PER_CHART authors so every label/value stays visible."""
        if n_items == 0:
            return row_cursor
        chunk = max(1, MAX_AUTHORS_PER_CHART)
        for start in range(0, n_items, chunk):
            end = min(start + chunk, n_items)
            size = end - start
            first_row = 1 + start          # +1 for the header row
            last_row = first_row + size - 1

            chart = workbook.add_chart({'type': 'bar'})
            chart.add_series({
                'name': series_name,
                'categories': [sheet_name, first_row, cat_col, last_row, cat_col],
                'values': [sheet_name, first_row, val_col, last_row, val_col],
                'data_labels': {'value': True, 'num_format': num_fmt, 'font': {'size': 8}},
                'gap': 40,
            })
            suffix = f' [{start + 1}-{end} of {n_items}]' if n_items > chunk else ''
            chart.set_title({'name': f'{chart_title}{suffix}'})
            chart.set_x_axis({'name': value_axis_name, 'major_gridlines': {'visible': True}})
            chart.set_y_axis({
                'name': 'Author',
                'reverse': True,          # descending data + reverse => largest bar at the TOP
                'interval_unit': 1,       # force a label for EVERY author
                'interval_tick': 1,
                'num_font': {'size': 8},
            })
            chart_height = max(300, size * CHART_ROW_HEIGHT_PX + 90)
            chart.set_size({'width': CHART_WIDTH_PX, 'height': chart_height})
            chart.set_legend({'none': True})
            ws.insert_chart(row_cursor, 0, chart)
            row_cursor += int(chart_height / 15) + 3
        return row_cursor

    # Chart 1: Author vs avg lead time hours (descending, all authors)
    current_chart_row = add_author_bar_charts(
        n_items=n_auth_lt, cat_col=0, val_col=1,
        series_name='Avg Lead Time (hours)',
        chart_title='Author vs Avg Lead Time (hours) - All Authors, Descending',
        value_axis_name='Hours', num_fmt='0.0',
        row_cursor=current_chart_row,
    )

    # Chart 2: Author vs MR count (descending, all authors)
    current_chart_row = add_author_bar_charts(
        n_items=n_auth_mr, cat_col=3, val_col=4,
        series_name='MR Count',
        chart_title='Author vs MR Count - All Authors, Descending',
        value_axis_name='Count', num_fmt='0',
        row_cursor=current_chart_row,
    )

    # Chart 3: Month vs avg lead time hours (line + trendline)
    if n_month > 0 and has_lt:
        chart2 = workbook.add_chart({'type': 'line'})
        chart2.add_series({
            'name': 'Avg Lead Time (hours)',
            'categories': [sheet_name, 1, month_offset_col, n_month, month_offset_col],
            'values': [sheet_name, 1, month_offset_col + 2, n_month, month_offset_col + 2],
            'data_labels': {'value': True, 'num_format': '0.0', 'font': {'size': 8}},
            'trendline': {'type': 'linear', 'display_equation': False},
            'marker': {'type': 'circle', 'size': 5},
        })
        chart2.set_title({'name': 'Monthly Avg Lead Time (hours) + Trend'})
        chart2.set_x_axis({'name': 'Month', 'interval_unit': 1, 'interval_tick': 1,
                           'num_font': {'size': 8, 'rotation': -45}})
        chart2.set_y_axis({'name': 'Hours'})
        chart2.set_legend({'none': True})
        chart2.set_size({'width': max(CHART_WIDTH_PX, 40 * max(n_month, 8)), 'height': 380})
        ws.insert_chart(current_chart_row, 0, chart2)
        current_chart_row += 27

    # Chart 4: Month vs MR count (line + trendline)
    if n_month > 0:
        chart4 = workbook.add_chart({'type': 'line'})
        chart4.add_series({
            'name': 'MR Count',
            'categories': [sheet_name, 1, month_offset_col, n_month, month_offset_col],
            'values': [sheet_name, 1, month_offset_col + 1, n_month, month_offset_col + 1],
            'data_labels': {'value': True, 'num_format': '0', 'font': {'size': 8}},
            'trendline': {'type': 'linear', 'display_equation': False},
            'marker': {'type': 'circle', 'size': 5},
        })
        chart4.set_title({'name': 'Monthly MR Count + Trend'})
        chart4.set_x_axis({'name': 'Month', 'interval_unit': 1, 'interval_tick': 1,
                           'num_font': {'size': 8, 'rotation': -45}})
        chart4.set_y_axis({'name': 'Count'})
        chart4.set_legend({'none': True})
        chart4.set_size({'width': max(CHART_WIDTH_PX, 40 * max(n_month, 8)), 'height': 380})
        ws.insert_chart(current_chart_row, 0, chart4)


# Get list of unique pods and crews
all_pods = sorted(pod_output_df['pod_name'].unique().tolist())
all_crews = sorted(pod_output_df['crew_name'].unique().tolist())

# For crew level, deduplicate MRs (each MR counted once per crew regardless of pods)
crew_dedup_cols = ['ID', 'IID', 'target_project_id', 'crew_name']
crew_dedup_cols = [c for c in crew_dedup_cols if c in pod_output_df.columns]
crew_df = pod_output_df.drop_duplicates(subset=crew_dedup_cols)

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    workbook = writer.book

    # Sheet 1: full enriched data (pod expanded)
    final_df.to_excel(writer, sheet_name='Data', index=False)

    # ---- Per-pod worksheets ----
    for pod in all_pods:
        pod_data = pod_output_df[pod_output_df['pod_name'] == pod]
        safe_name = re.sub(r'[\\/*?:\[\]]', '_', str(pod))[:28]
        sn = f'P_{safe_name}' if safe_name != 'N/A' else 'Pod_NA'
        if sn in writer.sheets:
            sn = sn[:28] + '_2'
        add_pod_charts(workbook, writer, sn, pod_data)

    # ---- Crew level worksheets ----
    for crew in all_crews:
        crew_data = crew_df[crew_df['crew_name'] == crew]
        safe_name = re.sub(r'[\\/*?:\[\]]', '_', str(crew))[:27]
        sn = f'C_{safe_name}' if safe_name != 'N/A' else 'Crew_NA'
        if sn in writer.sheets:
            sn = sn[:28] + '_2'
        add_pod_charts(workbook, writer, sn, crew_data)

print(f'Output written: {output_path}')
print(f'Total sheets: 1 (Data) + {len(all_pods)} pod sheets + {len(all_crews)} crew sheets')
print('Each sheet has stacked charts with data labels and trendlines')

## Section 12: Fast-Run Safeguards and Validation

In [ ]:
print('=' * 70)
print('RUN SUMMARY')
print('=' * 70)

stats = gl_client.stats()
print('API stats:')
print(f'  success: {stats["success"]}')
print(f'  failed:  {stats["failed"]}')
print(f'  cached:  {stats["cached"]}')

print('Data quality:')
print(f'  input rows: {len(df)}')
print(f'  output rows (pod-expanded): {len(pod_output_df)}')
print(f'  unique pods:  {pod_output_df["pod_name"].nunique()}')
print(f'  unique crews: {pod_output_df["crew_name"].nunique()}')
print(f'  project_name available: {(pod_output_df["project_name"] != "N/A").sum()}')
print(f'  author_name available:  {(pod_output_df["author_name"] != "N/A").sum()}')
print(f'  author_gpn available:   {(pod_output_df["author_gpn"] != "N/A").sum()}')
print(f'  pod_name available:     {(pod_output_df["pod_name"] != "N/A").sum()}')
print(f'  crew_name available:    {(pod_output_df["crew_name"] != "N/A").sum()}')

_unmatched = pod_output_df[pod_output_df['pod_name'] == 'N/A']
if len(_unmatched):
    print(f'  rows without a pod: {len(_unmatched)}')
    print('  top unmatched authors (name / gpn):')
    print(_unmatched[['author_name', 'author_gpn']].value_counts().head(10).to_string())

elapsed = elapsed_seconds()
speed = len(df) / elapsed if elapsed > 0 else 0.0
print(f'Elapsed seconds: {elapsed:.2f}')
print(f'Records per second: {speed:.2f}')
print('=' * 70)

## Optional: View Full Output Sample

In [ ]:
preview_cols = ['ID', 'IID', 'project_name', 'author_name', 'author_gpn',
                'pod_name', 'crew_name', 'merged_month_year', 'merged_date']
if LEAD_TIME_COL:
    preview_cols.append(LEAD_TIME_COL)
preview_cols = [c for c in preview_cols if c in pod_output_df.columns]
print(pod_output_df[preview_cols].head(15).to_string(index=False))

print('\nOutput dataframe shape:', pod_output_df.shape)
print('\nPod distribution:')
print(pod_output_df['pod_name'].value_counts().to_string())
print('\nCrew distribution:')
print(pod_output_df['crew_name'].value_counts().to_string())